# Análisis de Varianza (ANOVA) e Introducción al PCA

Este notebook introduce el **Análisis de Varianza (ANOVA)** como herramienta para comparar grupos y descomponer la variabilidad total de un conjunto de datos. A partir de esa base conceptual, se establece un puente natural hacia el **Análisis de Componentes Principales (PCA)**, que extiende la descomposición de la varianza al espacio multivariado.

## Objetivos
- Comprender la lógica de descomposición de varianza: total = entre grupos + dentro de grupos.
- Ejecutar e interpretar una ANOVA de un factor y de dos factores.
- Verificar los supuestos de ANOVA (normalidad, homocedasticidad, independencia).
- Entender cómo PCA generaliza la idea de «maximizar varianza explicada».
- Conectar el escree plot y la varianza explicada del PCA con la tabla ANOVA.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.stats import shapiro, levene, f_oneway
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(2025)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook')

print('Librerías cargadas correctamente.')

---
## Parte 1 — Análisis de Varianza (ANOVA)

### 1.1 Motivación: ¿Por qué no usar múltiples t-tests?

Cuando queremos comparar las medias de **tres o más grupos**, aplicar un t-test para cada par de grupos infla el error de tipo I (falsos positivos). Si tenemos $k$ grupos y nivel de significancia $\alpha$, la probabilidad de cometer al menos un error es:

$$P(\text{al menos un error}) = 1 - (1-\alpha)^{\binom{k}{2}}$$

Con $k = 4$ grupos y $\alpha = 0.05$:

$$P = 1 - (0.95)^{6} \approx 0.26 \quad \text{(¡26% de error global!)}$$

La ANOVA resuelve este problema comparando todos los grupos **simultáneamente** con una única prueba.

In [ ]:
# Ilustración del problema de comparaciones múltiples
k_values = np.arange(2, 11)
alpha = 0.05
error_global = [1 - (1 - alpha) ** (k * (k-1) // 2) for k in k_values]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, error_global, marker='o', color='steelblue', linewidth=2)
ax.axhline(0.05, color='tomato', linestyle='--', label='α = 0.05')
ax.set_xlabel('Número de grupos (k)')
ax.set_ylabel('Error tipo I global')
ax.set_title('Inflación del error con comparaciones múltiples')
ax.legend()
plt.tight_layout()
plt.show()

### 1.2 Descomposición de la varianza total

La idea central de ANOVA es descomponer la **suma de cuadrados total (SCT)** en dos partes:

$$\underbrace{\sum_{i=1}^{k}\sum_{j=1}^{n_i}(y_{ij} - \bar{y})^2}_{\text{SCT}} = \underbrace{\sum_{i=1}^{k} n_i(\bar{y}_i - \bar{y})^2}_{\text{SCE (entre grupos)}} + \underbrace{\sum_{i=1}^{k}\sum_{j=1}^{n_i}(y_{ij} - \bar{y}_i)^2}_{\text{SCD (dentro de grupos)}}
$$

| Fuente | SC | gl | CM | F |
|---|---|---|---|---|
| Entre grupos | SCE | $k-1$ | $\text{CME} = \frac{\text{SCE}}{k-1}$ | $F = \frac{\text{CME}}{\text{CMD}}$ |
| Dentro de grupos | SCD | $N-k$ | $\text{CMD} = \frac{\text{SCD}}{N-k}$ | — |
| Total | SCT | $N-1$ | — | — |

Si $H_0$ es verdadera (todas las medias son iguales), ambos cuadrados medios (CME y CMD) estiman la misma varianza poblacional $\sigma^2$, por lo que $F \approx 1$. Un $F$ grande indica que la variabilidad **entre grupos** es mayor de lo esperado por azar.

### 1.3 Datos de ejemplo: rendimiento de fertilizantes

Se tienen cuatro tratamientos de fertilizante (A, B, C, D) aplicados a parcelas agrícolas. La variable respuesta es el **rendimiento en kg/ha**.

In [ ]:
# Generación de datos sintéticos
n_por_grupo = 20
grupos = {
    'A': np.random.normal(4200, 300, n_por_grupo),
    'B': np.random.normal(4600, 280, n_por_grupo),
    'C': np.random.normal(4400, 320, n_por_grupo),
    'D': np.random.normal(5000, 310, n_por_grupo),
}

df = pd.DataFrame({
    'rendimiento': np.concatenate(list(grupos.values())),
    'fertilizante': np.repeat(list(grupos.keys()), n_por_grupo)
})

print(df.groupby('fertilizante')['rendimiento'].describe().round(1))

In [ ]:
# Diagrama de caja con media señalada
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='fertilizante', y='rendimiento', palette='Set2', ax=ax)
# Añadir medias
medias = df.groupby('fertilizante')['rendimiento'].mean()
for i, (grupo, m) in enumerate(medias.items()):
    ax.plot(i, m, marker='D', color='black', markersize=7, zorder=5)
ax.set_title('Rendimiento por tipo de fertilizante\n(diamante negro = media)')
ax.set_xlabel('Fertilizante')
ax.set_ylabel('Rendimiento (kg/ha)')
plt.tight_layout()
plt.show()

### 1.4 Verificación de supuestos

La ANOVA clásica requiere:
1. **Normalidad** de los residuos (por grupo).
2. **Homocedasticidad**: varianzas iguales en todos los grupos (prueba de Levene).
3. **Independencia** de las observaciones (diseño experimental).

In [ ]:
# --- Normalidad: Shapiro-Wilk por grupo ---
print('Prueba Shapiro-Wilk (H0: distribución normal)')
print('-' * 45)
for g, datos in grupos.items():
    stat, p = shapiro(datos)
    resultado = 'No se rechaza H0' if p > 0.05 else 'SE RECHAZA H0'
    print(f'  Grupo {g}: W={stat:.4f}, p={p:.4f}  →  {resultado}')

print()
# --- Homocedasticidad: Levene ---
stat_lev, p_lev = levene(*grupos.values())
resultado_lev = 'No se rechaza H0' if p_lev > 0.05 else 'SE RECHAZA H0'
print(f'Prueba de Levene (H0: varianzas iguales)')
print(f'  W={stat_lev:.4f}, p={p_lev:.4f}  →  {resultado_lev}')

In [ ]:
# Gráfico Q-Q de residuos globales
modelo_ols = smf.ols('rendimiento ~ C(fertilizante)', data=df).fit()
residuos = modelo_ols.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
axes[0].hist(residuos, bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de residuos')
axes[0].set_xlabel('Residuos')

# Q-Q plot
sm.qqplot(residuos, line='s', ax=axes[1], alpha=0.5)
axes[1].set_title('Q-Q plot de residuos')

plt.tight_layout()
plt.show()

### 1.5 ANOVA de un factor (one-way)

Con los supuestos verificados procedemos a realizar la prueba.

In [ ]:
# ANOVA con scipy
F, p_valor = f_oneway(*grupos.values())
print(f'F = {F:.4f},  p-valor = {p_valor:.6f}')
print('Conclusión:', 'Hay diferencias significativas entre grupos (α=0.05)' if p_valor < 0.05
      else 'No hay evidencia de diferencias significativas')

print()
# Tabla ANOVA completa con statsmodels
tabla_anova = sm.stats.anova_lm(modelo_ols, typ=1)
print(tabla_anova.round(4))

In [ ]:
# Descomposición visual de la varianza
SCT = ((df['rendimiento'] - df['rendimiento'].mean()) ** 2).sum()
SCE = sum(n_por_grupo * (m - df['rendimiento'].mean()) ** 2 for m in medias)
SCD = SCT - SCE

fig, ax = plt.subplots(figsize=(6, 4))
etiquetas = ['SCE\n(Entre grupos)', 'SCD\n(Dentro de grupos)']
valores = [SCE, SCD]
colores = ['#2ecc71', '#e74c3c']
bars = ax.bar(etiquetas, valores, color=colores, edgecolor='white', width=0.5)
for bar, v in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + SCT*0.01,
            f'{v/SCT:.1%}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Suma de cuadrados')
ax.set_title(f'Descomposición de la varianza total\nSCT = {SCT:,.0f}')
plt.tight_layout()
plt.show()

### 1.6 Comparaciones múltiples: prueba de Tukey HSD

La ANOVA nos dice **si** hay diferencias, pero no **entre cuáles** grupos. La prueba de Tukey controla el error familiar manteniendo $\alpha$ constante.

In [ ]:
tukey = pairwise_tukeyhsd(df['rendimiento'], df['fertilizante'], alpha=0.05)
print(tukey)

# Visualización
fig = tukey.plot_simultaneous(figsize=(8, 4))
plt.title('Intervalos de confianza simultáneos — Tukey HSD')
plt.tight_layout()
plt.show()

### 1.7 ANOVA de dos factores

Cuando hay **dos factores** de tratamiento (p.ej., fertilizante × riego) también podemos estudiar su **interacción**.

El modelo es:
$$y_{ijk} = \mu + \alpha_i + \beta_j + (\alpha\beta)_{ij} + \varepsilon_{ijk}$$

Generamos datos donde hay efecto de fertilizante, de riego y una leve interacción.

In [ ]:
# Datos dos factores: fertilizante (A,B,C,D) × riego (bajo, alto)
recs = []
efectos_fertil = {'A': 0, 'B': 400, 'C': 200, 'D': 800}
efectos_riego  = {'bajo': 0, 'alto': 300}
interaccion    = {('B', 'alto'): 200, ('D', 'bajo'): -150}  # interacciones específicas

for fert, ef in efectos_fertil.items():
    for riego, er in efectos_riego.items():
        inter = interaccion.get((fert, riego), 0)
        mu_ij = 4200 + ef + er + inter
        y = np.random.normal(mu_ij, 280, 15)
        for yi in y:
            recs.append({'rendimiento': yi, 'fertilizante': fert, 'riego': riego})

df2 = pd.DataFrame(recs)

# Tabla de medias
print(df2.groupby(['fertilizante', 'riego'])['rendimiento'].mean().unstack().round(1))

In [ ]:
# ANOVA de dos factores con interacción
modelo2 = smf.ols('rendimiento ~ C(fertilizante) * C(riego)', data=df2).fit()
tabla2 = sm.stats.anova_lm(modelo2, typ=2)
print(tabla2.round(4))

In [ ]:
# Gráfico de interacción
medias2 = df2.groupby(['fertilizante', 'riego'])['rendimiento'].mean().reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
for riego, sub in medias2.groupby('riego'):
    ax.plot(sub['fertilizante'], sub['rendimiento'], marker='o',
            linewidth=2, label=f'Riego: {riego}')
ax.set_title('Gráfico de interacción: Fertilizante × Riego')
ax.set_xlabel('Fertilizante')
ax.set_ylabel('Rendimiento medio (kg/ha)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Parte 2 — Del ANOVA al PCA: la varianza como hilo conductor

### 2.1 El concepto clave: maximizar varianza explicada

Tanto en ANOVA como en PCA el objetivo es **entender la variabilidad** de los datos:

| Aspecto | ANOVA | PCA |
|---|---|---|
| Pregunta | ¿Los grupos explican la varianza? | ¿Qué combinación lineal explica la máxima varianza? |
| Descomposición | SCT = SCE + SCD | $\text{Var total} = \lambda_1 + \lambda_2 + ... + \lambda_p$ |
| Resultado | Razón F, p-valor | Componentes principales, eigenvalores |
| Dimensión | 1 variable respuesta, $k$ grupos | $p$ variables, sin etiquetas de grupo |

> **Idea central del PCA**: en lugar de asignar varianza a factores discretos (como ANOVA), PCA encuentra las **direcciones** en el espacio $p$-dimensional a lo largo de las cuales los datos varían más.

### 2.2 Fundamentos matemáticos del PCA

Dado un conjunto de datos $\mathbf{X}$ de $n$ observaciones y $p$ variables (estandarizadas), PCA busca una matrix ortogonal $\mathbf{W}$ tal que las **componentes principales** $\mathbf{Z} = \mathbf{X}\mathbf{W}$ tengan varianza máxima y sean no correlacionadas.

Esto equivale a la **descomposición espectral** de la matriz de correlaciones $\mathbf{R}$:

$$\mathbf{R} = \mathbf{W}\boldsymbol{\Lambda}\mathbf{W}^T$$

donde $\boldsymbol{\Lambda} = \text{diag}(\lambda_1, \lambda_2, ..., \lambda_p)$ con $\lambda_1 \geq \lambda_2 \geq ... \geq \lambda_p \geq 0$.

**Proporción de varianza explicada** por el componente $j$:
$$PVE_j = \frac{\lambda_j}{\sum_{i=1}^{p}\lambda_i}$$

**Analogía con ANOVA**:  
$$\underbrace{\frac{\lambda_j}{\sum \lambda_i}}_{\text{PCA}} \longleftrightarrow \underbrace{\frac{\text{SCE}}{\text{SCT}}}_{\text{ANOVA}} = \eta^2 \quad (\text{eta cuadrado})$$

In [ ]:
# Dataset multivariado sintético con estructura grupal
n = 150
# 3 grupos con medias distintas en 5 variables
mu1 = [2, 3, 1, 4, 2]
mu2 = [5, 1, 4, 2, 5]
mu3 = [1, 5, 3, 1, 3]
cov = np.array([
    [1.0, 0.8, 0.3, 0.1, 0.5],
    [0.8, 1.0, 0.2, 0.3, 0.4],
    [0.3, 0.2, 1.0, 0.6, 0.1],
    [0.1, 0.3, 0.6, 1.0, 0.2],
    [0.5, 0.4, 0.1, 0.2, 1.0]
])

X1 = np.random.multivariate_normal(mu1, cov, n // 3)
X2 = np.random.multivariate_normal(mu2, cov, n // 3)
X3 = np.random.multivariate_normal(mu3, cov, n // 3)
X_all = np.vstack([X1, X2, X3])
grupos_mv = ['G1'] * (n//3) + ['G2'] * (n//3) + ['G3'] * (n//3)

cols = [f'X{i+1}' for i in range(5)]
df_mv = pd.DataFrame(X_all, columns=cols)
df_mv['grupo'] = grupos_mv
df_mv.head()

### 2.3 ANOVA univariado para cada variable

Antes de PCA, comprobemos qué variables discriminan significativamente entre grupos.

In [ ]:
resultados_anova = []
for col in cols:
    g1 = df_mv.loc[df_mv['grupo']=='G1', col]
    g2 = df_mv.loc[df_mv['grupo']=='G2', col]
    g3 = df_mv.loc[df_mv['grupo']=='G3', col]
    F, p = f_oneway(g1, g2, g3)
    # eta cuadrado
    grand_mean = df_mv[col].mean()
    sce = sum(len(g) * (g.mean() - grand_mean)**2 for g in [g1,g2,g3])
    sct = ((df_mv[col] - grand_mean)**2).sum()
    eta2 = sce / sct
    resultados_anova.append({'Variable': col, 'F': round(F,3), 'p-valor': round(p,4),
                              'η²': round(eta2, 3),
                              'Significativa': 'Sí' if p < 0.05 else 'No'})

df_anova_mv = pd.DataFrame(resultados_anova).set_index('Variable')
print(df_anova_mv)

In [ ]:
# Visualizar η² por variable
fig, ax = plt.subplots(figsize=(7, 4))
colores_bar = ['#2ecc71' if s == 'Sí' else '#e74c3c'
               for s in df_anova_mv['Significativa']]
ax.bar(df_anova_mv.index, df_anova_mv['η²'], color=colores_bar, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('η² (proporción de varianza explicada por grupo)')
ax.set_title('ANOVA univariado: η² por variable\n(verde = significativa, rojo = no significativa)')
plt.tight_layout()
plt.show()

### 2.4 Aplicación de PCA

Ahora aplicamos PCA al mismo conjunto de datos y analizamos cuánta varianza **total** captura cada componente.

In [ ]:
# Estandarización
scaler = StandardScaler()
X_std = scaler.fit_transform(df_mv[cols])

# PCA
pca = PCA()
Z = pca.fit_transform(X_std)

# Eigenvalores (varianza de cada componente)
eigenvalores = pca.explained_variance_
pve = pca.explained_variance_ratio_
pve_acum = np.cumsum(pve)

tabla_pca = pd.DataFrame({
    'Eigenvalor (λ)': eigenvalores.round(3),
    'PVE (%)': (pve * 100).round(2),
    'PVE Acumulado (%)': (pve_acum * 100).round(2)
}, index=[f'PC{i+1}' for i in range(len(eigenvalores))])
print(tabla_pca)

In [ ]:
# Scree plot + varianza acumulada
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scree plot
axes[0].bar(range(1, len(eigenvalores)+1), eigenvalores, color='steelblue', edgecolor='white')
axes[0].axhline(1, color='tomato', linestyle='--', label='Criterio Kaiser (λ=1)')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Eigenvalor (varianza explicada)')
axes[0].set_title('Scree Plot')
axes[0].legend()

# Varianza acumulada
axes[1].plot(range(1, len(pve_acum)+1), pve_acum * 100, marker='o', color='steelblue', linewidth=2)
axes[1].axhline(80, color='tomato', linestyle='--', label='80% varianza')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].set_title('Varianza explicada acumulada')
axes[1].legend()
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

### 2.5 Cargas del PCA (loadings)

Las cargas ($\mathbf{W}$) indican qué tanto contribuye cada variable original a cada componente.

$$w_{jk} = \text{correlación entre la variable } X_j \text{ y el componente } PC_k$$

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=cols,
    columns=[f'PC{i+1}' for i in range(len(eigenvalores))]
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(loadings.iloc[:, :3], annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Cargas (loadings) — Primeros 3 componentes principales')
ax.set_ylabel('Variable original')
plt.tight_layout()
plt.show()

### 2.6 Biplot: scores + loadings 

El biplot proyecta las observaciones (scores) y las variables (loadings) en el plano PC1–PC2 simultáneamente.

In [ ]:
scores = Z[:, :2]  # PC1 y PC2
load2d = pca.components_[:2].T  # cargas de las 2 primeras CP

colores_grupo = {'G1': '#e74c3c', 'G2': '#2ecc71', 'G3': '#3498db'}

fig, ax = plt.subplots(figsize=(9, 7))

for g, color in colores_grupo.items():
    mask = np.array(grupos_mv) == g
    ax.scatter(scores[mask, 0], scores[mask, 1],
               c=color, label=g, alpha=0.6, s=50)

# Flechas de cargas (escaladas)
escala = 3.5
for i, var in enumerate(cols):
    ax.annotate('', xy=(load2d[i, 0]*escala, load2d[i, 1]*escala),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.8))
    ax.text(load2d[i, 0]*escala*1.1, load2d[i, 1]*escala*1.1,
            var, fontsize=11, fontweight='bold', ha='center')

ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel(f'PC1 ({pve[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pve[1]*100:.1f}% varianza)')
ax.set_title('Biplot PCA: Scores (puntos) y Loadings (flechas)')
ax.legend(title='Grupo')
plt.tight_layout()
plt.show()

### 2.7 Relación ANOVA sobre componentes principales

Una vez obtenidas las componentes, podemos aplicar ANOVA sobre ellas para saber cuáles discriminan entre grupos. Esto es especialmente útil cuando los grupos son conocidos.

In [ ]:
# ANOVA de cada CP con respecto a los grupos
df_scores = pd.DataFrame(Z, columns=[f'PC{i+1}' for i in range(Z.shape[1])])
df_scores['grupo'] = grupos_mv

res_pca_anova = []
for pc in [f'PC{i+1}' for i in range(5)]:
    g1 = df_scores.loc[df_scores['grupo']=='G1', pc]
    g2 = df_scores.loc[df_scores['grupo']=='G2', pc]
    g3 = df_scores.loc[df_scores['grupo']=='G3', pc]
    F, p = f_oneway(g1, g2, g3)
    grand = df_scores[pc].mean()
    sce = sum(len(g)*(g.mean()-grand)**2 for g in [g1,g2,g3])
    sct = ((df_scores[pc]-grand)**2).sum()
    eta2 = sce/sct
    res_pca_anova.append({'Componente': pc, 'F': round(F,3),
                          'p-valor': round(p,4), 'η²': round(eta2,3)})

df_ra = pd.DataFrame(res_pca_anova).set_index('Componente')
print(df_ra)

In [ ]:
# Comparación: PVE del PCA vs η² sobre los componentes
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(5)
width = 0.35
ax.bar(x - width/2, pve * 100, width, label='PVE (varianza total explicada por PCA)', color='steelblue')
ax.bar(x + width/2, df_ra['η²'] * 100, width, label='η² (varianza de CP explicada por grupo)', color='#e67e22')
ax.set_xticks(x)
ax.set_xticklabels([f'PC{i+1}' for i in range(5)])
ax.set_ylabel('%')
ax.set_title('PVE del PCA vs η² por componente (ANOVA)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Resumen y puntos clave

### ANOVA
- Descompone la varianza total en varianza **entre grupos** (explicada) y **dentro de grupos** (error).
- La razón $F = \text{CME}/\text{CMD}$ mide cuánto mayor es la variabilidad entre grupos respecto al ruido.
- Los supuestos son normalidad de residuos, homocedasticidad e independencia.
- Cuando se rechaza $H_0$, las pruebas post-hoc (Tukey, Bonferroni) identifican qué pares difieren.

### PCA como extensión
- PCA busca las **direcciones de máxima varianza** en el espacio $p$-dimensional.
- Los eigenvalores $\lambda_j$ cumplen el mismo rol que SCE/SCT en ANOVA: cuantifican la «importancia» de cada componente.
- El **scree plot** es el análogo visual de la tabla ANOVA.
- Las cargas (loadings) revelan qué variables originales explican cada componente.
- El biplot combina la posición de muestras y variables en el espacio reducido.

### Conexión práctica
| Concepto ANOVA | Equivalente PCA |
|---|---|
| SC total ($\sum \lambda_j$) | Varianza total de los datos |
| SC entre grupos por variable | Eigenvalor $\lambda_j$ del $j$-ésimo CP |
| $\eta^2 = \text{SCE}/\text{SCT}$ | PVE$_j = \lambda_j / \sum \lambda$ |
| Factor (variable categórica) | Componente principal (combinación lineal) |
| Grado de libertad | Dimensión capturada |

> **El PCA extiende la lógica de ANOVA del espacio univariado al multivariado: en lugar de preguntar "¿cuánto explica el grupo?", pregunta "¿cuánto explica esta dirección del espacio?".**